# 01 - Ingesta de viajes: descompresión y capa bronze

**Antes de este notebook.** Databricks no llega a los servidores de la IM, así que los archivos se suben desde una computadora local con `python scripts/subir_viajes.py`. Quedan en el volume como `viajes_zip/viajes_stm_AAAA_MM.zip`.

**Parte A. Descompresión.** Extrae cada ZIP del volume a `viajes/viajes_stm_AAAA_MM.csv`.

**Parte B. Bronze.** Convierte cada mes a la tabla Delta `viajes_bronze`, sin limpiar ni transformar los datos: todas las columnas se leen como texto y se agregan columnas de trazabilidad.

Ambas partes son **idempotentes**: los meses ya descomprimidos o ya cargados se saltean, salvo que se fuerce el reproceso.

In [0]:
%run ./00_setup.py

In [0]:
import os
import socket
import subprocess
import urllib.request

import requests

API_PATH = "/api/3/action/package_show?id=viajes-realizados-en-los-omnibus-del-sistema-de-transporte-metropolitano-stm"
T = 8  # segundos por intento
resultados = []

def registrar(prueba, ok, detalle=""):
    resultados.append((prueba, ok, detalle))
    print(f"{'OK   ' if ok else 'FALLA'}  {prueba:<55} {detalle}")

# ---------------------------------------------------------------------------
print("=== 0. Entorno de red de Databricks ===")
for var in ["HTTP_PROXY", "HTTPS_PROXY", "http_proxy", "https_proxy", "NO_PROXY", "no_proxy"]:
    if os.environ.get(var):
        print(f"  {var} = {os.environ[var]}")
print("  (si no se imprimió nada, no hay proxy configurado por variables de entorno)\n")

# ---------------------------------------------------------------------------
print("=== 1. DNS: IPv4 e IPv6 ===")
HOSTS = ["ckan.montevideo.gub.uy", "ckan-data.montevideo.gub.uy",
         "imnube.montevideo.gub.uy", "montevideo.gub.uy", "catalogodatos.gub.uy"]
ips = {}
for h in HOSTS:
    try:
        infos = socket.getaddrinfo(h, 443)
        v4 = sorted({i[4][0] for i in infos if i[0] == socket.AF_INET})
        v6 = sorted({i[4][0] for i in infos if i[0] == socket.AF_INET6})
        ips[h] = (v4, v6)
        registrar(f"DNS {h}", True, f"IPv4={v4} IPv6={v6}")
    except Exception as e:
        registrar(f"DNS {h}", False, type(e).__name__)

# ---------------------------------------------------------------------------
print("\n=== 2. Conexión TCP por host, puerto e IP ===")
for h, (v4, v6) in ips.items():
    for puerto in (443, 80):
        for ip in v4[:1] + v6[:1]:
            fam = socket.AF_INET6 if ":" in ip else socket.AF_INET
            try:
                s = socket.socket(fam, socket.SOCK_STREAM)
                s.settimeout(T)
                s.connect((ip, puerto))
                s.close()
                registrar(f"TCP {h}:{puerto} ({'v6' if fam == socket.AF_INET6 else 'v4'})", True, ip)
            except Exception as e:
                registrar(f"TCP {h}:{puerto} ({'v6' if fam == socket.AF_INET6 else 'v4'})", False, type(e).__name__)

# ---------------------------------------------------------------------------
print("\n=== 3. Variantes de headers (requests, HTTPS) ===")
HEADERS = {
    "sin headers": {},
    "Mozilla corto (notebook anterior)": {
        "User-Agent": "Mozilla/5.0",
        "Accept": "text/csv,application/zip,application/octet-stream,text/plain,*/*"},
    "Chrome completo": {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                      "(KHTML, like Gecko) Chrome/130.0 Safari/537.36",
        "Accept": "application/json,text/html,*/*;q=0.8",
        "Accept-Language": "es-UY,es;q=0.9",
        "Connection": "keep-alive"},
    "Chrome + Referer del portal": {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) Chrome/130.0",
        "Referer": "https://ckan.montevideo.gub.uy/",
        "Accept-Language": "es-UY,es;q=0.9"},
    "curl UA": {"User-Agent": "curl/8.5.0"},
}
for nombre, hdr in HEADERS.items():
    try:
        r = requests.get(f"https://ckan.montevideo.gub.uy{API_PATH}", headers=hdr, timeout=T)
        registrar(f"headers: {nombre}", r.ok, f"HTTP {r.status_code}")
    except Exception as e:
        registrar(f"headers: {nombre}", False, type(e).__name__)

# ---------------------------------------------------------------------------
print("\n=== 4. Protocolo y host ===")
for url in [f"http://ckan.montevideo.gub.uy{API_PATH}",
            f"https://ckan-data.montevideo.gub.uy{API_PATH}",
            f"https://catalogodatos.gub.uy{API_PATH}"]:
    try:
        r = requests.get(url, headers=HEADERS["Chrome completo"], timeout=T, allow_redirects=True)
        registrar(f"GET {url.split('/api')[0]}", r.ok, f"HTTP {r.status_code}")
    except Exception as e:
        registrar(f"GET {url.split('/api')[0]}", False, type(e).__name__)

# ---------------------------------------------------------------------------
print("\n=== 5. Otros clientes y configuración de proxy ===")
URL = f"https://ckan.montevideo.gub.uy{API_PATH}"

try:
    s = requests.Session()
    s.trust_env = False  # ignora cualquier proxy del entorno
    r = s.get(URL, headers=HEADERS["Chrome completo"], timeout=T)
    registrar("requests sin proxy del entorno", r.ok, f"HTTP {r.status_code}")
except Exception as e:
    registrar("requests sin proxy del entorno", False, type(e).__name__)

try:
    req = urllib.request.Request(URL, headers=HEADERS["Chrome completo"])
    with urllib.request.urlopen(req, timeout=T) as resp:
        registrar("urllib (biblioteca estándar)", True, f"HTTP {resp.status}")
except Exception as e:
    registrar("urllib (biblioteca estándar)", False, type(e).__name__)

for flag, nombre in [("-4", "curl IPv4"), ("-6", "curl IPv6"), ("--http1.1", "curl HTTP/1.1")]:
    try:
        out = subprocess.run(
            ["curl", flag, "-sS", "-o", "/dev/null", "-w", "%{http_code}",
             "--connect-timeout", str(T), "-A", HEADERS["Chrome completo"]["User-Agent"], URL],
            capture_output=True, text=True, timeout=T + 5)
        codigo = out.stdout.strip()
        registrar(nombre, codigo.startswith("2"), f"HTTP {codigo} {out.stderr.strip()[:60]}")
    except Exception as e:
        registrar(nombre, False, type(e).__name__)

# ---------------------------------------------------------------------------
print("\n=== RESUMEN ===")
exitos = [p for p, ok, _ in resultados if ok and not p.startswith("DNS")]
print(f"{len(exitos)} combinaciones con éxito (sin contar DNS):")
for p in exitos:
    print("  ", p)

In [0]:
import socket

for host in ["pypi.org", "ckan.montevideo.gub.uy", "montevideo.gub.uy"]:
    try:
        ip = socket.gethostbyname(host)
        with socket.create_connection((ip, 443), timeout=15):
            print(f"Conexión TCP OK      {host} ({ip})")
    except socket.timeout:
        print(f"TIMEOUT en TCP       {host}")
    except Exception as e:
        print(f"FALLA {type(e).__name__:<12} {host}")
        

## Parámetros

| Parámetro | Valor | Efecto |
|---|---|---|
| `MESES` | `["2024-01"]` | Procesa solo esos meses. **Usar un mes en la primera ejecución.** |
| `MESES` | `None` | Procesa todos los meses subidos al volume. |
| `DESCOMPRIMIR` | `False` | Saltea la Parte A (si los CSV ya están en el volume). |
| `FORZAR_BRONZE` | `True` | Vuelve a cargar en bronze meses que ya estaban cargados. |

In [0]:
MESES = ["2024-01"]      # None = todos los meses subidos al volume
DESCOMPRIMIR = True
FORZAR_BRONZE = False

In [0]:
from pathlib import Path
 
from pyspark.sql import functions as F
 
from bondis import descarga, ingesta
 
importlib.reload(descarga)
importlib.reload(ingesta)
 
 
def mes_elegido(anio_mes):
    """True si el mes 'AAAA-MM' entra en el filtro MESES."""
    return MESES is None or anio_mes in MESES

## Parte A. Descompresión
### A.1 ZIP disponibles en el volume

In [0]:
if DESCOMPRIMIR:
    zips = sorted(Path(config.RAW_VIAJES_ZIP).glob("viajes_stm_*.zip"))
    zips_por_mes = {am: ps for am, ps in ingesta.agrupar_por_mes(zips).items() if mes_elegido(am)}
 
    print(f"Meses a descomprimir: {len(zips_por_mes)}")
    for am, ps in zips_por_mes.items():
        print(f"  {am}  <-  {Path(ps[0]).name}")
else:
    print("Descompresión desactivada (DESCOMPRIMIR = False).")

### A.2 Extracción de los CSV

In [0]:
fallidos = []
 
if DESCOMPRIMIR:
    for am, ps in zips_por_mes.items():
        try:
            descarga.descomprimir_mes(ps[0], config.RAW_VIAJES)
        except Exception as e:  # se registra y se sigue con el próximo mes
            fallidos.append((am, str(e)))
            print(f"[FAIL] {am}: {e}")
 
    print(f"\nFallidos: {len(fallidos)}")
    for am, err in fallidos:
        print(f"  {am}: {err}")

## Parte B. Carga a bronze
### B.1 Inventario de archivos y estado de la tabla

In [0]:
archivos = sorted(
    Path(config.RAW_VIAJES).glob("viajes_stm_*.csv")
)
por_mes = {am: ps for am, ps in ingesta.agrupar_por_mes(archivos).items() if mes_elegido(am)}
 
tabla_existe = spark.catalog.tableExists(config.TABLE_BRONZE)
cargados = set()
if tabla_existe:
    cargados = {
        r["anio_mes"]
        for r in spark.table(config.TABLE_BRONZE).select("anio_mes").distinct().collect()
    }
 
pendientes = {am: ps for am, ps in por_mes.items() if FORZAR_BRONZE or am not in cargados}
 
print(f"Meses con CSV en el volume: {len(por_mes)}")
print(f"Meses ya cargados en bronze: {len(cargados & set(por_mes))}")
print(f"Meses a cargar ahora:        {len(pendientes)}  {list(pendientes)}")

### B.2 Carga mes a mes

Para cada mes:
1. Detecta separador y codificación mirando el inicio del archivo.
2. Lee el CSV con **todas las columnas como texto** (bronze no interpreta tipos).
3. Normaliza los nombres de columnas.
4. Agrega `anio_mes`, `archivo_origen` y `fecha_ingesta`.
5. Reemplaza **solo la partición de ese mes** en la tabla (`replaceWhere`).

In [0]:
columnas_por_mes = {}
 
for am, paths in pendientes.items():
    sep, enc, encabezado = ingesta.detectar_formato(paths[0])
    print(f"[{am}] {len(paths)} archivo/s · separador {sep!r} · {enc}")
 
    crudo = (
        spark.read
        .option("header", True)
        .option("sep", sep)
        .option("encoding", enc)
        .option("inferSchema", False)
        .csv(paths)
    )
 
    nuevos_nombres = ingesta.normalizar_columnas(crudo.columns)
    df = crudo.select(
        *[F.col(f"`{c}`").alias(n) for c, n in zip(crudo.columns, nuevos_nombres)],
        F.lit(am).alias("anio_mes"),
        F.col("_metadata.file_name").alias("archivo_origen"),
        F.current_timestamp().alias("fecha_ingesta"),
    )
 
    escritor = df.write.format("delta").mode("overwrite").option("mergeSchema", "true")
    if tabla_existe:
        escritor.option("replaceWhere", f"anio_mes = '{am}'").saveAsTable(config.TABLE_BRONZE)
    else:
        escritor.partitionBy("anio_mes").saveAsTable(config.TABLE_BRONZE)
        tabla_existe = True
 
    filas = spark.table(config.TABLE_BRONZE).where(F.col("anio_mes") == am).count()
    columnas_por_mes[am] = nuevos_nombres
    print(f"        {filas:,} filas · columnas: {nuevos_nombres}")

### B.3 Control de consistencia de columnas

Si la IM cambió el formato entre años, distintos meses tendrán columnas distintas.
Bronze las conserva todas (`mergeSchema`); la unificación se resuelve en silver.

In [0]:
if columnas_por_mes:
    esquemas = {}
    for am, cols in columnas_por_mes.items():
        esquemas.setdefault(tuple(cols), []).append(am)
    if len(esquemas) == 1:
        print("Todos los meses cargados tienen las mismas columnas.")
    else:
        print(f"ATENCIÓN: {len(esquemas)} esquemas distintos.")
        for cols, meses in esquemas.items():
            print(f"  Meses {meses}:\n    {list(cols)}")
else:
    print("No se cargaron meses en esta ejecución.")

## Verificación

In [0]:
if spark.catalog.tableExists(config.TABLE_BRONZE):
    display(
        spark.table(config.TABLE_BRONZE)
        .groupBy("anio_mes")
        .agg(F.count("*").alias("filas"), F.countDistinct("archivo_origen").alias("archivos"))
        .orderBy("anio_mes")
    )

In [0]:
if spark.catalog.tableExists(config.TABLE_BRONZE):
    spark.table(config.TABLE_BRONZE).printSchema()
    display(spark.table(config.TABLE_BRONZE).limit(10))